## **Build a ReAct Agent**

You're a software engineer on a mission: build an AI agent that doesn't just respond—it thinks. In this lab, you'll step into the role of an AI architect, designing a smart assistant that solves tough problems by reasoning through them and taking purposeful actions.

Using the ReAct (Reasoning + Acting) framework, you'll teach your agent to think step by step, consult tools like search engines or calculators, and adapt on the fly. It’s not just about answers—it’s about how the agent gets there.

By the end of the lab, your AI will face a mystery that can’t be solved with knowledge alone. It will need logic, resourcefulness, and the ability to act—just like you, the engineer who built it.

## What is ReAct?

**ReAct** stands for **Reasoning + Acting**. It's a framework that combines:

1. **Reasoning**: The agent thinks through problems step by step, maintaining an internal dialogue about what it needs to do.
2. **Acting**: The agent can use external tools (search engines, calculators, APIs) to gather information or perform actions.
3. **Observing**: The agent processes the results from its actions and incorporates them into its reasoning.

This creates a powerful loop: **Think → Act → Observe → Think → Act → ...**

### Why ReAct Matters

Traditional language models are limited by their training data cutoff and can't access real-time information. ReAct agents overcome this by:
- Accessing current information through web searches
- Performing calculations with specialized tools
- Breaking down complex problems into manageable steps
- Adapting their approach based on intermediate results


In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults
import os
import dotenv

C:\Users\Vish\AppData\Local\Temp\ipykernel_25332\2498070253.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [3]:
dotenv.load_dotenv()
taily_key = os.getenv("travily_api_key")
groq_key = os.getenv("groq_api")

In [15]:
tavily_search = TavilySearchResults(tavily_api_key=taily_key, max_results=3)

In [5]:
from langchain_groq import ChatGroq

d:\Projects\langgraph-explore\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_key
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B83D30C390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B83D3C1B50>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [17]:
tavily_search.invoke("What is the weather in Colombo now?")

[{'title': 'Colombo weather in July 2026 | Colombo 14 day weather',
  'url': 'https://www.weather25.com/asia/sri-lanka/western/colombo?page=month&month=July',
  'content': 'weather25.com \n\nUnited States England Australia Canada\n\n°F °C\n\nWeather in July 2026\n\nRemove from your favorite locations Add to my locations\n\nShare\n\n1. Home\n2. Asia\n3. weather in Sri LankaSri Lanka\n4. Western\n5. Colombo\n6. July\n\nLocation was added to My Locations\n\nLocation was removed from My Locations\n\n# Colombo weather in July 2026\n\nClick on a day for an hourly weather forecast\n\nJul 17\n\nLight rain shower\n\n3 mm\n\n29° / 26°Jul 18\n\nPatchy rain possible\n\n0.8 mm\n\n29° / 26°Jul 19\n\nPatchy light drizzle\n\n13.5 mm\n\n28° / 26°Monday\n\nJul 20\n\nPatchy rain possible\n\n4.5 mm\n\n28° / 26°Tuesday\n\nJul 21\n\nPatchy rain possible\n\n2.3 mm\n\n29° / 26°Wednesday\n\nJul 22\n\nLight rain shower\n\n4 mm\n\n29° / 26°Thursday\n\nJul 23\n\nLight rain shower\n\n10 mm\n\n28° / 26°Friday\n\nJu

In [8]:
from langchain_core.tools import tool

In [18]:
@tool
def clothing_recommendation(weather:str) -> str:
    
    """Returns clothing recommendation based on the given weather condition
    
    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.
    
    Args:
    - weather (str): A brief description of the weather (e.g., "Overcast, 64.9°F")
    
    Return:
    - str: A string with clothing recommendations suitable for the weather
    """
    
    weather = weather.lower()
    
    if "snow" in weather or "freezig" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."


In [19]:
@tool
def search_tool(query:str):
    """
    Take a string as a input and search the web for information using Tavily API

    Args:
        query (str): The search query string
    
    Returns:
        str: Search results related to query
    """
    
    results = tavily_search.invoke(query)
    return results

In [20]:
tools = [search_tool, clothing_recommendation]

tools_map = {
    tool.name:tool for tool in tools
}

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage,SystemMessage,ToolMessage

In [22]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
    """You are a helpful assistant that thinks step-by-step and use tools when needed.
    
    when responding to query:
    1. First, think about what information you need.
    2. Use available tools if you need current data or specific capabilities
    3. Provide clear, helpful response based on your reasoning and any tool results
    """
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [23]:
llm_chain = chat_prompt | llm.bind_tools(tools=tools) 

In [29]:
from typing import TypedDict, Annotated, Sequence
from langgraph.graph.message import add_messages

In [30]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [31]:
state: AgentState = {"messages": []}

# append a message using the reducer properly
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Hi")])
print("After greeting:", state["messages"])

# add another message (e.g. a question)
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Weather in NYC?")])
print("After question:", state)

After greeting: [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='190c9adc-84de-4df2-90ae-7cae7635a06d')]
After question: {'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='190c9adc-84de-4df2-90ae-7cae7635a06d'), HumanMessage(content='Weather in NYC?', additional_kwargs={}, response_metadata={}, id='ba4e353d-5c95-4986-a784-5e7363cf9012')]}


In [32]:
def model_call(state:AgentState):
    
    results = llm_chain.invoke({
        "messages": state['messages']
    })
    
    return {
        "messages": [results]
    }
    

In [ ]:
def tool_call(state:AgentState):
    
    last_ai_message = state['messages'][-1]
    tools_call = last_ai_message.tool_calls
    
    results = []
    
    for tool_use in tools_call:
        tool_result = tools_map[tool_use['name']].invoke(tool_use['args'])
        results.append(
            ToolMessage(
                content=tool_result,
                name=tool_use['name'],
                tool_call_d = tool_use["id"]
            )
        )
    
    return {
        "messages": results
    }

In [34]:
from langgraph.graph import END

In [35]:
def should_continue(state:AgentState):
    
    last_ai_message = state['messages'][-1]
    
    if not last_ai_message.tool_calls:
        return "end"
    
    else:
        return "continue"